In [0]:
import pandas as pd
import numpy as np

# Actual monthly averages of EIA daily spot prices (same source STEO forecasts)
actual = spark.sql("""
    SELECT series_id, date_trunc('month', price_date) AS month, AVG(price) AS actual
    FROM oil_shock.gold.fct_price_daily
    WHERE series_id IN ('RBRTE', 'RWTC')
      AND price_date < date_trunc('month', current_date())   -- complete months only
    GROUP BY series_id, date_trunc('month', price_date)
""").toPandas()
actual["month"] = pd.to_datetime(actual["month"])
actual["steo_id"] = actual["series_id"].map({"RBRTE": "BREPUUS", "RWTC": "WTIPUUS"})

steo = spark.sql("""
    SELECT release_month, series_id, target_month, value AS forecast,
           CAST(ROUND(months_ahead) AS INT) AS horizon, source
    FROM oil_shock.gold.fct_steo_forecast
    WHERE series_id IN ('BREPUUS', 'WTIPUUS') AND is_forecast
""").toPandas()
steo["target_month"] = pd.to_datetime(steo["target_month"])
steo["release_date"] = pd.to_datetime(steo["release_month"] + "-01")
steo = steo[steo["horizon"].between(0, 12)]

# Join actuals for the target month
a = actual.rename(columns={"steo_id": "series"})[["series", "month", "actual"]]
df = steo.merge(a, left_on=["series_id", "target_month"], right_on=["series", "month"], how="inner")

# No-change baseline: last complete month's average before the release
prev = a.rename(columns={"month": "base_month", "actual": "baseline"})
df["base_month"] = df["release_date"] - pd.offsets.MonthBegin(1)
df = df.merge(prev, left_on=["series_id", "base_month"], right_on=["series", "base_month"], how="left",
              suffixes=("", "_b"))
df = df.dropna(subset=["baseline"])

df["steo_err_pct"] = 100 * (df["forecast"] - df["actual"]) / df["actual"]
df["base_err_pct"] = 100 * (df["baseline"] - df["actual"]) / df["actual"]
df["steo_better"] = df["steo_err_pct"].abs() < df["base_err_pct"].abs()
df["period"] = np.where(df["target_month"] >= "2026-03-01", "war", "pre_war")

summary = (df.groupby(["series_id", "period", "horizon"])
           .agg(n=("steo_err_pct", "size"),
                steo_mape=("steo_err_pct", lambda x: x.abs().mean()),
                baseline_mape=("base_err_pct", lambda x: x.abs().mean()),
                steo_bias=("steo_err_pct", "mean"),
                baseline_bias=("base_err_pct", "mean"),
                steo_beats_baseline_pct=("steo_better", lambda x: 100 * x.mean()))
           .reset_index().round(1))

print("Detail rows:", len(df), "| releases:", df["release_month"].nunique(),
      "| target months:", df["target_month"].min().date(), "to", df["target_month"].max().date())
show = summary[(summary["series_id"] == "BREPUUS") & (summary["horizon"].isin([0, 1, 2, 3, 6, 12]))]
display(show.sort_values(["period", "horizon"]))

In [0]:
import mlflow
from datetime import datetime, timezone

run_stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

detail = df[["series_id", "release_month", "target_month", "horizon", "period", "source",
             "forecast", "actual", "baseline", "steo_err_pct", "base_err_pct", "steo_better"]].copy()
detail["target_month"] = detail["target_month"].dt.date
detail["run_at_utc"] = run_stamp
summary_out = summary.copy()
summary_out["run_at_utc"] = run_stamp

for name, out in {"steo_track_record": detail, "steo_accuracy_summary": summary_out}.items():
    (spark.createDataFrame(out)
         .write.mode("overwrite").option("overwriteSchema", "true")
         .saveAsTable(f"oil_shock.ml.{name}"))
    print("Saved oil_shock.ml." + name, "rows:", len(out))

mlflow.set_experiment("/Users/zulhaq2@gmail.com/oil-shock-forecast-comparison")
with mlflow.start_run(run_name=f"steo_track_record {run_stamp}"):
    mlflow.set_tags({"notebook": "notebooks/04_ml/42_forecast_comparison", "project": "oil-shock-2026",
                     "author": "Zia Malik", "step": "EIA track record vs no-change"})
    mlflow.log_params({"actual": "monthly average of EIA daily spot (RBRTE, RWTC)",
                       "baseline": "no change: last complete month before release",
                       "war_start_target": "2026-03", "horizons": "0-12 months"})
    for _, r in summary[summary["series_id"] == "BREPUUS"].iterrows():
        mlflow.log_metrics({f"brent_{r['period']}_h{r['horizon']}_steo_mape": float(r["steo_mape"]),
                            f"brent_{r['period']}_h{r['horizon']}_baseline_mape": float(r["baseline_mape"]),
                            f"brent_{r['period']}_h{r['horizon']}_steo_beats_pct": float(r["steo_beats_baseline_pct"])})
    path = "/tmp/steo_accuracy_summary.csv"
    summary.to_csv(path, index=False)
    mlflow.log_artifact(path)
print("MLflow run logged.")

In [0]:
from sklearn.ensemble import HistGradientBoostingRegressor

feat = spark.table("oil_shock.gold.fct_model_features_daily").toPandas()
feat["price_date"] = pd.to_datetime(feat["price_date"])
feat = feat.sort_values("price_date").reset_index(drop=True)
feat["month"] = feat["price_date"].dt.to_period("M")

# Derived features (all use data up to that day only)
feat["ret_20d"] = np.log(feat["brent"] / feat["brent"].shift(20))
feat["vs_ma250"] = np.log(feat["brent"] / feat["brent"].rolling(250, min_periods=120).mean())
feat["usd_chg_1m"] = np.log(feat["usd_index"] / feat["usd_index"].shift(21))
feat["y10_chg_1m"] = feat["treasury_10y_yield"] - feat["treasury_10y_yield"].shift(21)
feat["stocks_chg_4w"] = feat["crude_stocks_commercial_kbbl"] - feat["crude_stocks_commercial_kbbl"].shift(20)

monthly = feat.groupby("month")["brent"].mean()
current_month = pd.Timestamp.today().to_period("M")
complete = monthly[monthly.index < current_month]

FEATS = ["mtd_vs_base", "base_vs_prev", "ret_20d", "brent_ret_5d_past", "vs_ma250", "brent_vol_20d",
         "brent_wti_spread", "diesel_crack", "usd_chg_1m", "y10_chg_1m", "vix_model_only",
         "brent_crowding_z", "wti_crowding_z", "stocks_chg_4w", "china_cli_change_3m", "us_cli_change_3m",
         "hormuz_tanker_pct", "gulf_risk_level_index"]

rows = []
for m in pd.period_range("2016-01", current_month, freq="M"):
    day10 = m.to_timestamp() + pd.Timedelta(days=9)
    after = feat[feat["price_date"] >= day10]
    if after.empty or after.iloc[0]["month"] != m:
        continue
    o = after.iloc[0]
    if (m - 1) not in complete.index or (m - 2) not in complete.index:
        continue
    base = complete[m - 1]
    mtd = feat[(feat["month"] == m) & (feat["price_date"] <= o["price_date"])]["brent"].mean()
    r = {"origin_month": m, "origin_date": o["price_date"], "baseline": base,
         "mtd_vs_base": np.log(mtd / base), "base_vs_prev": np.log(base / complete[m - 2])}
    for c in FEATS:
        if c not in r:
            r[c] = o[c]
    for h in (1, 2):
        tm = m + h
        r[f"actual_h{h}"] = complete[tm] if tm in complete.index else np.nan
        r[f"y_h{h}"] = np.log(r[f"actual_h{h}"] / base) if tm in complete.index else np.nan
    rows.append(r)
panel = pd.DataFrame(rows)
print("Forecast origins:", len(panel), "|", panel["origin_month"].min(), "to", panel["origin_month"].max())

def gbm(loss, q=None):
    kw = dict(max_depth=2, learning_rate=0.05, max_iter=200, min_samples_leaf=10, random_state=42)
    return HistGradientBoostingRegressor(loss=loss, quantile=q, **kw) if q else HistGradientBoostingRegressor(loss=loss, **kw)

preds = []
for h in (1, 2):
    for k, row in panel.iterrows():
        if row["origin_month"] < pd.Period("2018-01", "M") or np.isnan(row[f"y_h{h}"]):
            continue
        train = panel[(panel["origin_month"] + h <= row["origin_month"] - 1) & panel[f"y_h{h}"].notna()]
        if len(train) < 24:
            continue
        X, y = train[FEATS].astype(float), train[f"y_h{h}"]
        x0 = row[FEATS].astype(float).to_frame().T
        mid = gbm("squared_error").fit(X, y).predict(x0)[0]
        lo = gbm("quantile", 0.1).fit(X, y).predict(x0)[0]
        hi = gbm("quantile", 0.9).fit(X, y).predict(x0)[0]
        lo, mid, hi = sorted([lo, mid, hi])
        b, act = row["baseline"], row[f"actual_h{h}"]
        preds.append({"origin_month": str(row["origin_month"]), "horizon": h,
                      "target_month": str(row["origin_month"] + h), "train_n": len(train),
                      "baseline": b, "actual": act, "ours_mid": b * np.exp(mid),
                      "ours_low": b * np.exp(lo), "ours_high": b * np.exp(hi)})
bt = pd.DataFrame(preds)

# EIA forecasts for the same release month and horizon
steo_tr = spark.sql("""
    SELECT release_month AS origin_month, horizon, forecast AS steo
    FROM oil_shock.ml.steo_track_record WHERE series_id = 'BREPUUS'
""").toPandas()
bt = bt.merge(steo_tr, on=["origin_month", "horizon"], how="left")

for c in ["ours_mid", "baseline", "steo"]:
    bt[f"{c}_ape"] = 100 * (bt[c] - bt["actual"]).abs() / bt["actual"]
bt["ours_beats_base"] = bt["ours_mid_ape"] < bt["baseline_ape"]
bt["in_80_range"] = (bt["actual"] >= bt["ours_low"]) & (bt["actual"] <= bt["ours_high"])
bt["period"] = np.where(bt["target_month"] >= "2026-03", "war", "pre_war")

def summarize(g):
    s = g.dropna(subset=["steo"])
    return pd.Series({"n": len(g), "ours_mape": g["ours_mid_ape"].mean(), "baseline_mape": g["baseline_ape"].mean(),
                      "steo_mape_same_months": s["steo_ape"].mean(), "n_with_steo": len(s),
                      "ours_beats_base_pct": 100 * g["ours_beats_base"].mean(),
                      "range_80_coverage_pct": 100 * g["in_80_range"].mean()})

overall = bt.groupby("horizon").apply(summarize).round(1).reset_index()
by_period = bt.groupby(["horizon", "period"]).apply(summarize).round(1).reset_index()

for _, r in overall.iterrows():
    verdict = "PASSES" if r["ours_mape"] < r["baseline_mape"] else "FAILS"
    print(f"Kill rule, {int(r['horizon'])} month(s) ahead: ours {r['ours_mape']}% vs no-change {r['baseline_mape']}% -> {verdict}")
display(overall)
display(by_period)

In [0]:
import mlflow
from datetime import datetime, timezone

run_stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
fv = feat[["price_date", "brent", "brent_vol_20d"]].copy()

def asof(date):
    s = fv[fv["price_date"] <= date]
    return s.iloc[-1]

# ---------- 1) Add the random-walk benchmark (latest price at the forecast date) ----------
spot = {str(r["origin_month"]): fv.loc[fv["price_date"] == r["origin_date"], "brent"].iloc[0]
        for _, r in panel.iterrows()}
bt["random_walk"] = bt["origin_month"].map(spot)
bt["random_walk_ape"] = 100 * (bt["random_walk"] - bt["actual"]).abs() / bt["actual"]

comp_rows = []
for (h, per), g in list(bt.groupby(["horizon", "period"])) + [((h, "all"), g) for h, g in bt.groupby("horizon")]:
    comp_rows.append({"horizon": h, "period": per, "n": len(g),
                      "ours_gbm_mape": g["ours_mid_ape"].mean(), "no_change_mape": g["baseline_ape"].mean(),
                      "random_walk_mape": g["random_walk_ape"].mean(), "eia_steo_mape": g["steo_ape"].mean(),
                      "ours_range_80_coverage": 100 * g["in_80_range"].mean()})
comparison = pd.DataFrame(comp_rows).round(1).sort_values(["horizon", "period"])

# ---------- 2) 80% range around EIA's forecast, from EIA's own past misses ----------
tr = spark.table("oil_shock.ml.steo_track_record").where("series_id = 'BREPUUS'").toPandas()
tr["target_month"] = pd.to_datetime(tr["target_month"])
tr["release_start"] = pd.to_datetime(tr["release_month"] + "-01")
tr["v"] = tr["release_month"].map(lambda r: asof(pd.Timestamp(r + "-10"))["brent_vol_20d"])
tr["z"] = np.log(tr["actual"] / tr["forecast"]) / tr["v"]

fan_rows = []
for _, row in tr[tr["release_start"] >= "2018-01-01"].iterrows():
    hist = tr[(tr["horizon"] == row["horizon"]) & (tr["target_month"] < row["release_start"])]
    if len(hist) < 24:
        continue
    q10, q90 = hist["z"].quantile([0.1, 0.9])
    lo, hi = row["forecast"] * np.exp(q10 * row["v"]), row["forecast"] * np.exp(q90 * row["v"])
    fan_rows.append({"release_month": row["release_month"], "horizon": int(row["horizon"]),
                     "target_month": row["target_month"].date(), "eia_forecast": row["forecast"],
                     "low_80": lo, "high_80": hi, "actual": row["actual"],
                     "in_range": lo <= row["actual"] <= hi,
                     "period": "war" if row["target_month"] >= pd.Timestamp("2026-03-01") else "pre_war"})
fan_bt = pd.DataFrame(fan_rows)
fan_cov = (fan_bt.groupby(["horizon", "period"])["in_range"]
           .agg(n="size", coverage_pct=lambda x: 100 * x.mean()).round(1).reset_index())

# Current range around the latest EIA release
latest = spark.sql("SELECT MAX(release_month) AS m FROM oil_shock.gold.fct_steo_forecast").first()["m"]
cur = spark.sql(f"""
    SELECT release_month, CAST(ROUND(months_ahead) AS INT) AS horizon, target_month, value AS eia_forecast
    FROM oil_shock.gold.fct_steo_forecast
    WHERE series_id = 'BREPUUS' AND is_forecast AND release_month = '{latest}'
""").toPandas()
cur = cur[cur["horizon"].between(0, 12)].sort_values("horizon")
v_now = asof(pd.Timestamp(latest + "-10"))["brent_vol_20d"]
bands = tr.groupby("horizon")["z"].quantile([0.1, 0.9]).unstack()
cur["low_80"] = cur.apply(lambda r: r["eia_forecast"] * np.exp(bands.loc[r["horizon"], 0.1] * v_now), axis=1)
cur["high_80"] = cur.apply(lambda r: r["eia_forecast"] * np.exp(bands.loc[r["horizon"], 0.9] * v_now), axis=1)
cur["vol_used"] = v_now
cur = cur.round(2)

# ---------- 3) Save and log ----------
bt_out = bt.copy()
tables = {"forecast_backtest": bt_out, "forecast_comparison_summary": comparison,
          "steo_range_backtest": fan_bt, "steo_range_current": cur}
for name, out in tables.items():
    out = out.copy()
    out["run_at_utc"] = run_stamp
    (spark.createDataFrame(out)
         .write.mode("overwrite").option("overwriteSchema", "true")
         .saveAsTable(f"oil_shock.ml.{name}"))
    print("Saved oil_shock.ml." + name, "rows:", len(out))

ov = comparison[comparison["period"] == "all"].set_index("horizon")
mlflow.set_experiment("/Users/zulhaq2@gmail.com/oil-shock-forecast-comparison")
with mlflow.start_run(run_name=f"forecast_backtest {run_stamp}"):
    mlflow.set_tags({"notebook": "notebooks/04_ml/42_forecast_comparison", "project": "oil-shock-2026",
                     "author": "Zia Malik", "kill_rule": "GBM must beat no-change at 1 and 2 months",
                     "kill_rule_outcome": "TRIGGERED: GBM lost; EIA forecast + historical 80% range used"})
    mlflow.log_params({"model": "HistGradientBoosting (depth 2, 200 iters) + quantile 0.1/0.9",
                       "origins": "10th of each month, 2018-01 onward, expanding window",
                       "range_method": "EIA forecast x exp(quantile of past log misses / vol x current vol)"})
    for h in (1, 2):
        mlflow.log_metrics({f"h{h}_ours_mape": float(ov.loc[h, "ours_gbm_mape"]),
                            f"h{h}_no_change_mape": float(ov.loc[h, "no_change_mape"]),
                            f"h{h}_random_walk_mape": float(ov.loc[h, "random_walk_mape"]),
                            f"h{h}_eia_mape": float(ov.loc[h, "eia_steo_mape"]),
                            f"h{h}_ours_range_coverage": float(ov.loc[h, "ours_range_80_coverage"])})
    mlflow.log_metric("eia_range_coverage_all", float(100 * fan_bt["in_range"].mean()))
    for name in ["forecast_comparison_summary", "steo_range_current"]:
        p = f"/tmp/{name}.csv"
        tables[name].to_csv(p, index=False)
        mlflow.log_artifact(p)
print("MLflow run logged.")

print("\nForecast comparison (average miss, % of actual):")
display(comparison)
print("EIA 80% range, tested on past releases (should be ~80%):")
display(fan_cov[fan_cov["horizon"].isin([0, 1, 2, 3, 6, 12])])
print(f"Current range around EIA's {latest} forecast:")
display(cur[["target_month", "horizon", "eia_forecast", "low_80", "high_80"]])

In [0]:
import pandas as pd
import numpy as np
import mlflow
from datetime import datetime, timezone

run_stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

fv = spark.table("oil_shock.gold.fct_model_features_daily").select("price_date", "brent_vol_20d").toPandas()
fv["price_date"] = pd.to_datetime(fv["price_date"])
fv = fv.sort_values("price_date").dropna()
v_long = fv.loc[fv["price_date"] < "2026-01-01", "brent_vol_20d"].median()

def asof_v(d):
    return fv.loc[fv["price_date"] <= d, "brent_vol_20d"].iloc[-1]

tr = spark.table("oil_shock.ml.steo_track_record").where("series_id = 'BREPUUS'").toPandas()
tr["target_month"] = pd.to_datetime(tr["target_month"])
tr["release_start"] = pd.to_datetime(tr["release_month"] + "-01")
tr["v"] = tr["release_month"].map(lambda r: asof_v(pd.Timestamp(r + "-10")))
tr["logmiss"] = np.log(tr["actual"] / tr["forecast"])

def eff_vol(v, h, hl):
    w = 1.0 if np.isinf(hl) else 0.5 ** (h / hl)
    return np.sqrt(w * v ** 2 + (1 - w) * v_long ** 2)

def backtest(hl):
    out = []
    for _, row in tr[tr["release_start"] >= "2018-01-01"].iterrows():
        h = int(row["horizon"])
        hist = tr[(tr["horizon"] == h) & (tr["target_month"] < row["release_start"])]
        if len(hist) < 24:
            continue
        z = hist["logmiss"] / eff_vol(hist["v"].values, h, hl)
        q10, q90 = np.quantile(z, [0.1, 0.9])
        ev = eff_vol(row["v"], h, hl)
        lo, hi = row["forecast"] * np.exp(q10 * ev), row["forecast"] * np.exp(q90 * ev)
        out.append({"release_month": row["release_month"], "horizon": h, "target_month": row["target_month"].date(),
                    "eia_forecast": row["forecast"], "low_80": lo, "high_80": hi, "actual": row["actual"],
                    "in_range": lo <= row["actual"] <= hi,
                    "width_pct": 100 * (hi - lo) / row["forecast"],
                    "period": "war" if row["target_month"] >= pd.Timestamp("2026-03-01") else "pre_war"})
    return pd.DataFrame(out)

calib_rows, results = [], {}
for hl in [1, 2, 3, 6, np.inf]:
    b = backtest(hl)
    results[hl] = b
    pre = b[b["period"] == "pre_war"]
    cov_h = pre.groupby("horizon")["in_range"].mean() * 100
    calib_rows.append({"half_life_months": "none (old method)" if np.isinf(hl) else hl,
                       "pre_war_coverage_avg": round(cov_h.mean(), 1),
                       "avg_gap_from_80": round((cov_h - 80).abs().mean(), 1),
                       "avg_width_pct": round(pre["width_pct"].mean(), 1),
                       "war_coverage": round(100 * b.loc[b["period"] == "war", "in_range"].mean(), 1)})
calib = pd.DataFrame(calib_rows)
best_hl = [1, 2, 3, 6, np.inf][int(calib["avg_gap_from_80"].idxmin())]
best = results[best_hl]
print("Chosen half-life (months):", best_hl, "| long-run daily vol:", round(v_long, 4))

# Current range with the chosen method
latest = spark.sql("SELECT MAX(release_month) AS m FROM oil_shock.gold.fct_steo_forecast").first()["m"]
cur = spark.sql(f"""
    SELECT release_month, CAST(ROUND(months_ahead) AS INT) AS horizon, target_month, value AS eia_forecast
    FROM oil_shock.gold.fct_steo_forecast
    WHERE series_id = 'BREPUUS' AND is_forecast AND release_month = '{latest}'
""").toPandas()
cur = cur[cur["horizon"].between(0, 12)].sort_values("horizon").reset_index(drop=True)
v_now = asof_v(pd.Timestamp(latest + "-10"))
lows, highs = [], []
for _, r in cur.iterrows():
    h = int(r["horizon"])
    hist = tr[tr["horizon"] == h]
    z = hist["logmiss"] / eff_vol(hist["v"].values, h, best_hl)
    q10, q90 = np.quantile(z, [0.1, 0.9])
    ev = eff_vol(v_now, h, best_hl)
    lows.append(r["eia_forecast"] * np.exp(q10 * ev))
    highs.append(r["eia_forecast"] * np.exp(q90 * ev))
cur["low_80"], cur["high_80"] = lows, highs
cur["half_life_months"] = float(best_hl)
cur = cur.round(2)

# Save (replaces the earlier range tables) and log
tables = {"steo_range_backtest": best, "steo_range_current": cur, "steo_range_calibration": calib}
for name, out in tables.items():
    out = out.copy()
    out["run_at_utc"] = run_stamp
    (spark.createDataFrame(out.astype({c: str for c in out.columns if out[c].dtype == object}))
         .write.mode("overwrite").option("overwriteSchema", "true")
         .saveAsTable(f"oil_shock.ml.{name}"))
    print("Saved oil_shock.ml." + name, "rows:", len(out))

mlflow.set_experiment("/Users/zulhaq2@gmail.com/oil-shock-forecast-comparison")
with mlflow.start_run(run_name=f"eia_range_calibrated {run_stamp}"):
    mlflow.set_tags({"notebook": "notebooks/04_ml/42_forecast_comparison", "project": "oil-shock-2026",
                     "author": "Zia Malik", "step": "EIA 80% range with mean-reverting volatility"})
    mlflow.log_params({"half_life_candidates": "1, 2, 3, 6 months, none",
                       "chosen_half_life_months": str(best_hl), "long_run_daily_vol": round(float(v_long), 5),
                       "selection": "closest pre-war coverage to 80% across horizons"})
    mlflow.log_metrics({"pre_war_coverage_avg": float(calib.loc[calib['avg_gap_from_80'].idxmin(), 'pre_war_coverage_avg']),
                        "war_coverage": float(calib.loc[calib['avg_gap_from_80'].idxmin(), 'war_coverage'])})
    p = "/tmp/steo_range_calibration.csv"
    calib.to_csv(p, index=False)
    mlflow.log_artifact(p)
print("MLflow run logged.")

display(calib)
display(cur[["target_month", "horizon", "eia_forecast", "low_80", "high_80"]])

In [0]:
import time
t0 = time.time()
ALPHA = 0.2   # 80% range

def interval_score(b):
    lo, hi, y = b["low_80"], b["high_80"], b["actual"]
    s = (hi - lo) + (2 / ALPHA) * (lo - y).clip(lower=0) + (2 / ALPHA) * (y - hi).clip(lower=0)
    return 100 * s / y   # as % of the actual price, lower is better

score_rows = []
for hl, b in results.items():
    pre = b[b["period"] == "pre_war"].copy()
    pre["score"] = interval_score(pre)
    score_rows.append({"half_life_months": "none (old method)" if np.isinf(hl) else hl,
                       "interval_score_pre_war": round(pre["score"].mean(), 1),
                       "coverage_pre_war": round(100 * pre["in_range"].mean(), 1),
                       "width_pct_pre_war": round(pre["width_pct"].mean(), 1)})
    print(f"half-life {hl}: score {score_rows[-1]['interval_score_pre_war']}  ({time.time() - t0:.0f}s)")
scores = pd.DataFrame(score_rows)
hl_list = list(results.keys())
best_hl = hl_list[int(scores["interval_score_pre_war"].idxmin())]
print("Chosen half-life (months):", best_hl)

# Recompute the current range with the chosen half-life
lows, highs = [], []
for _, r in cur.iterrows():
    h = int(r["horizon"])
    hist = tr[tr["horizon"] == h]
    z = hist["logmiss"] / eff_vol(hist["v"].values, h, best_hl)
    q10, q90 = np.quantile(z, [0.1, 0.9])
    ev = eff_vol(v_now, h, best_hl)
    lows.append(r["eia_forecast"] * np.exp(q10 * ev))
    highs.append(r["eia_forecast"] * np.exp(q90 * ev))
cur["low_80"], cur["high_80"] = np.round(lows, 2), np.round(highs, 2)
cur["half_life_months"] = float(best_hl)

calib2 = calib.merge(scores, on="half_life_months", how="left")
tables = {"steo_range_backtest": results[best_hl], "steo_range_current": cur, "steo_range_calibration": calib2}
for name, out in tables.items():
    out = out.copy()
    out["run_at_utc"] = run_stamp
    (spark.createDataFrame(out.astype({c: str for c in out.columns if out[c].dtype == object}))
         .write.mode("overwrite").option("overwriteSchema", "true")
         .saveAsTable(f"oil_shock.ml.{name}"))
    print("Saved oil_shock.ml." + name, "rows:", len(out))

with mlflow.start_run(run_name=f"eia_range_interval_score {run_stamp}"):
    mlflow.set_tags({"notebook": "notebooks/04_ml/42_forecast_comparison", "project": "oil-shock-2026",
                     "author": "Zia Malik", "step": "half-life chosen by interval score (width + miss penalty)"})
    mlflow.log_params({"selection": "lowest pre-war interval score, alpha 0.2",
                       "chosen_half_life_months": str(best_hl)})
    mlflow.log_metric("chosen_interval_score", float(scores["interval_score_pre_war"].min()))
print(f"Done in {time.time() - t0:.0f}s")

display(calib2)
display(cur[["target_month", "horizon", "eia_forecast", "low_80", "high_80"]])